# AMPS I · Week 12 — HW11: solver iterations and time
**Due:** before the Week 13 lecture · **Points:** 100 · **Submit:** this notebook (`.ipynb`) with all outputs

**Rules**
- Before submitting: *Runtime → Restart session and run all*. Notebooks that do not run end to end receive 0.
- An error inside a `%%fenicsx` cell does **not** stop *Run all*. Read every output before you submit.
- LLM assistants are encouraged. Fill in the **AI-usage log** at the end.
- Every number you report must come from a cell you ran. Times: KSP setup + solve, measured after the forms exist; every timed solve must have a positive
  converged reason **and** agree with a direct solve.
- Timings vary between runs. Compare growth over a wide range of sizes and orders of magnitude, not the third digit.

| Question | Topic | Points |
|---|---|:---:|
| Q1 | Preconditioned spectra: Stokes with $\hat S\ne S$, the triangular preconditioner, Darcy | 20 |
| Q2 | 3D lid-driven Stokes cavity: LU vs two block preconditioners | 40 |
| Q3 | Darcy with a high-contrast permeability | 20 |
| Q4 | LLM debugging: a solver benchmark with four bugs | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

## Q1 — Theory (20 pts)

$\mathscr K=\begin{pmatrix}A&B^\top\\B&0\end{pmatrix}$, $A$ SPD, $B$ of full row rank, $S=BA^{-1}B^\top$.

**(a) (8)** Let $\hat S$ be SPD and $\sigma_1,\dots,\sigma_m$ the eigenvalues of $\hat S^{-1}S$. Show that every eigenvalue of $\mathscr P^{-1}\mathscr K$ with
$\mathscr P=\mathrm{diag}(A,\hat S)$ is $1$ or $\frac{1\pm\sqrt{1+4\sigma_i}}{2}$. For Stokes with viscosity $\mu$ and the **unscaled** $\hat S=M_p$: in which interval do the
negative eigenvalues lie (use $\beta_h$ and §A4)? What happens as $\mu\to0$, and what does that mean for MINRES?

**(b) (6)** $\mathscr P_U=\begin{pmatrix}A&B^\top\\0&-S\end{pmatrix}$. Compute $\mathscr K\mathscr P_U^{-1}$ and show $(\mathscr K\mathscr P_U^{-1}-I)^2=0$. Why does right-preconditioned GMRES then
converge in at most two iterations? Why is MINRES not an option with $\mathscr P_U$?

**(c) (6) Darcy.** $A=M_\sigma$ (RT1 mass), $B$ from $-\int v\,\nabla\cdot\boldsymbol\sigma$, $M_u$ the DG0 mass matrix, $T=B^\top M_u^{-1}B$. Show that with
$\mathscr P=\mathrm{diag}(M_\sigma+T,\,M_u)$ every eigenvalue of $\mathscr P^{-1}\mathscr K$ is $1$ or $\lambda=-\tau/(1+\tau)$ with $\tau>0$ an eigenvalue of $M_\sigma^{-1}T$.
Why are these clustered near $-1$ on fine meshes? (For RT1/DG0, $\nabla\cdot V_h\subset W_h$ gives $T_{ij}=\int\nabla\cdot\boldsymbol\phi_j\,\nabla\cdot\boldsymbol\phi_i$.)

> *your answers to (a)–(c)*

## Q2 — 3D lid-driven Stokes cavity: LU vs block preconditioners (40 pts)

Unit cube, Taylor–Hood, $\mu=1$, $\mathbf f=0$; $\mathbf u=(1,0,0)$ on the lid $z=1$, $\mathbf u=0$ on the other five faces. Pressure constant: a null space.

**(a) (15)** Write `cavity3d(N)` that returns the blocked operators (as in the lab's `stokes_blocked`, but with this geometry and these boundary conditions) and a
solver function. Solve with (i) LU (MUMPS with null-pivot detection), (ii) MINRES + `additive` fieldsplit (GAMG on $A$ with block size 3,
Jacobi on $M_p$), (iii) GMRES + Schur `upper` (`a11`, same blocks). rtol $10^{-8}$. For $N=4,8,12,16$ print DOFs, iterations, reason, time, and
$|x_{it}-x_{LU}|/|x_{LU}|$ for (ii) and (iii). (About 1.5 minutes on the build machine, LU at $N=16$ dominating; longer on Colab.)

**(b) (10)** Plot time vs DOFs (log–log, three solvers) and report the slope of each curve over the last three points. At which $N$ does LU stop being the fastest?

**(c) (15)** Explain (i) why the iteration counts of (ii) and (iii) are almost but not exactly independent of $N$, (ii) why the LU slope is larger in 3D than in
2D (lecture §C4), and (iii) estimate from your slopes the LU and MINRES times at $N=32$ ($\approx$ 860 k DOFs). Which other resource limits LU before time does?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(a) cavity3d, three solvers, N = 4, 8, 12, 16; save the table to /tmp/fnx/hw11_q2.npy for (b)

from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

In [ ]:
# TODO: Q2(b) log-log plot of time vs DOFs, slopes over the last three points

> *your answers to (b) and (c)*

## Q3 — Darcy with a high-contrast permeability (20 pts)

$K(\mathbf x)=1$ on the white and $\kappa$ on the black squares of a $4\times4$ checkerboard; $f=1$; $u=0$ on $\partial\Omega$ (natural); RT1 × DG0.
The flux equation carries $K^{-1}$: $\int K^{-1}\boldsymbol\sigma\cdot\boldsymbol\tau-\int u\,\nabla\cdot\boldsymbol\tau=0$.

**(a) (12)** Extend the lab's `darcy` to a DG0 coefficient $K$ and run, for $\kappa=1,10^2,10^4$ and $N=32,64$: (i) MINRES + $\mathrm{diag}(M_{\sigma,K},M_u)$,
(ii) MINRES + $\mathrm{diag}(M_{\sigma,K}+(\nabla\cdot,\nabla\cdot),M_u)$, (iii) GMRES + Schur `upper` with `selfp` (Jacobi on block 0, BoomerAMG on the Schur block),
where $M_{\sigma,K}$ is the $K^{-1}$-weighted RT mass matrix. `ksp_max_it` 2000 (keeps the cell to a few minutes on Colab). Table of iterations (with reason when negative).

**(b) (8)** Which preconditioners are robust in $h$ **and** $\kappa$? Explain why `selfp` copes with the jumps (what is $B\,\mathrm{diag}(M_{\sigma,K})^{-1}B^\top$ in terms of $K$?), and
why the L² preconditioner gets worse with $\kappa$.

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a) Darcy with a checkerboard permeability

from mpi4py import MPI
import time
import numpy as np
import ufl
from petsc4py import PETSc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import LinearProblem, apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def wall(t0):
    """Elapsed time, maximum over ranks (call on all ranks)."""
    return comm.allreduce(time.perf_counter() - t0, op=MPI.MAX)

> *your table and answers to (b)*

## Q4 — LLM debugging: a solver benchmark with four bugs (20 pts)

An LLM was asked: *"DOLFINx 0.11, -np 2: benchmark a direct solver against MINRES with a block-diagonal fieldsplit preconditioner for 2D Taylor–Hood Stokes
(the usual MMS) at N = 16, 32, 64; print the iteration counts and the times."* It produced the cell below (run it as given).

**(a) (12)** Find the four bugs. For each: the line, what is wrong, how it shows (or why it does not show) in the output, and how you detected it.
Add your own checks — reason, $\|\mathbf u-\mathbf u_h\|$, $|x_{it}-x_{LU}|/|x_{LU}|$, `-np 1` vs `-np 2` — in a copy of the cell.

**(b) (5)** Corrected cell: iterations, reasons, errors (both solvers must reach the discretisation error of Week 11: $9.7\times10^{-5}$, $1.2\times10^{-5}$, $1.5\times10^{-6}$), times.

**(c) (3)** Which of the four bugs would you **not** see with `-np 1`? Which one makes the iterative solver look *faster* than it is?

In [ ]:
%%fenicsx -np 2
from mpi4py import MPI
from petsc4py import PETSc
import time
import numpy as np
import ufl
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD

def setup(N):
    msh = mesh.create_unit_square(comm, N, N)
    V = fem.functionspace(msh, ("Lagrange", 2, (2,)))
    Q = fem.functionspace(msh, ("Lagrange", 1))
    u, p = ufl.TrialFunction(V), ufl.TrialFunction(Q)
    v, q = ufl.TestFunction(V), ufl.TestFunction(Q)
    x = ufl.SpatialCoordinate(msh)
    u_ex = ufl.as_vector([ufl.sin(ufl.pi * x[0]) * ufl.cos(ufl.pi * x[1]), -ufl.cos(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])])
    p_ex = ufl.sin(ufl.pi * x[0]) * ufl.sin(ufl.pi * x[1])
    f = -ufl.div(ufl.grad(u_ex)) + ufl.grad(p_ex)
    a = fem.form([[ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, -p * ufl.div(v) * ufl.dx],
                  [-q * ufl.div(u) * ufl.dx, None]])
    a_p = fem.form([[ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx, None], [None, p * q * ufl.dx]])
    L = fem.form([ufl.inner(f, v) * ufl.dx, ufl.ZeroBaseForm((q,))])
    u_bc = fem.Function(V); u_bc.interpolate(fem.Expression(u_ex, V.element.interpolation_points))
    msh.topology.create_connectivity(1, 2)
    bc = fem.dirichletbc(u_bc, fem.locate_dofs_topological(V, 1, mesh.exterior_facet_indices(msh.topology)))
    A = assemble_matrix(a, bcs=[bc]); A.assemble()
    P = assemble_matrix(a_p, bcs=[bc]); P.assemble()
    b = assemble_vector(L, kind=PETSc.Vec.Type.MPI)
    apply_lifting(b, a, bcs=fem.bcs_by_block(fem.extract_function_spaces(a, 1), [bc]))
    b.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
    set_bc(b, fem.bcs_by_block(fem.extract_function_spaces(L), [bc]))
    nu = V.dofmap.index_map.size_local * 2
    off = V.dofmap.index_map.local_range[0] * 2
    is_u = PETSc.IS().createStride(nu, off, 1, comm=comm)
    is_p = PETSc.IS().createStride(Q.dofmap.index_map.size_local, off + nu, 1, comm=comm)
    z = A.createVecLeft(); z.array[nu:] = 1.0; z.normalize()
    nsp = PETSc.NullSpace().create(vectors=[z]); A.setNullSpace(nsp); nsp.remove(b)
    return msh, V, A, P, b, is_u, is_p

for N in (16, 32, 64):
    msh, V, A, P, b, is_u, is_p = setup(N)

    t0 = time.perf_counter()
    lu = PETSc.KSP().create(comm)
    lu.setOperators(A, P)
    lu.setType("preonly"); lu.getPC().setType("lu"); lu.getPC().setFactorSolverType("mumps")
    lu.getPC().setFactorSetUpSolverType(); lu.getPC().getFactorMatrix().setMumpsIcntl(icntl=24, ival=1)
    x_lu = A.createVecRight(); lu.solve(b, x_lu)
    t_lu = time.perf_counter() - t0

    t0 = time.perf_counter()
    it = PETSc.KSP().create(comm)
    it.setOperators(A, P)
    it.setType("cg"); it.setTolerances(rtol=1e-3)
    pc = it.getPC(); pc.setType("fieldsplit"); pc.setFieldSplitIS(("u", is_u), ("p", is_p))
    pc.setFieldSplitType(PETSc.PC.CompositeType.ADDITIVE); pc.setUp()
    ku, kp = pc.getFieldSplitSubKSP()
    ku.setType("preonly"); ku.getPC().setType("hypre"); kp.setType("preonly"); kp.getPC().setType("jacobi")
    x_it = A.createVecRight(); it.solve(b, x_it)
    t_it = time.perf_counter() - t0

    if comm.rank == 0:
        print(f"N = {N:3d}: LU {t_lu:.3f} s | iterative {it.getIterationNumber():4d} its, {t_it:.3f} s")

In [ ]:
%%fenicsx -np 2
# TODO: Q4 your diagnostic copy and the corrected cell

> *your table for (a) and answers to (b), (c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …